# SMS Spam Detection — Beginner-Friendly ML Project

**Goal:** predict whether a message is spam (`1`) or ham / not spam (`0`) using the numeric features in `spam_detection_dataset.csv`.

> **Important dataset note:** this CSV contains numeric features (`num_links`, `num_words`, `has_offer`, `sender_score`, `all_caps`) and the target `is_spam`. It does **not** contain the actual SMS message text. Therefore, this version uses **Logistic Regression classification**, not TF-IDF text vectorization. To build a true text-based TF-IDF project, you need a dataset with a message-text column.

Run the notebook from top to bottom, one cell at a time.

## Step 1 — Install the required libraries

In VS Code, open **Terminal → New Terminal** and run:

```bash
python -m pip install -r requirements.txt
```

If the libraries are already installed, you can continue to Step 2. The libraries used are explained in `README.md`.

## Step 2 — Import libraries
Imports bring tools into Python so we can use them.
import os

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

import joblib
from pathlib import Path

RANDOM_STATE = 42

## Step 3 — Load the CSV dataset
Keep the CSV in the same folder as this notebook.

In [ ]:
DATA_PATH = Path("spam_detection_dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Put spam_detection_dataset.csv in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Rows and columns:", df.shape)
display(df.head())

## Step 4 — Understand the data
Each row is one example. `is_spam` is the answer the model learns to predict.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("\nTarget counts (0 = ham, 1 = spam):")
print(df["is_spam"].value_counts(dropna=False))

## Step 5 — Validate and prepare data
We check that the expected columns exist and remove rows with missing values. We do not include `is_spam` among the input features.

In [ ]:
FEATURES = ["num_links", "num_words", "has_offer", "sender_score", "all_caps"]
TARGET = "is_spam"

missing_columns = [col for col in FEATURES + [TARGET] if col not in df.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing these expected columns: {missing_columns}")

df = df[FEATURES + [TARGET]].dropna().copy()

# Target should contain only 0 (ham) and 1 (spam).
if not set(df[TARGET].unique()).issubset({0, 1}):
    raise ValueError("The is_spam column must contain only 0 and 1.")

X = df[FEATURES]
y = df[TARGET].astype(int)

print("Input feature shape:", X.shape)
print("Target shape:", y.shape)
print("Rows remaining after removing missing values:", len(df))

## Step 6 — Split into training and testing data
The model learns from 80% of the data. The remaining 20% is held back to test performance on examples it did not train on.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))
print("\nSpam proportion in full data:", round(y.mean(), 3))
print("Spam proportion in test data:", round(y_test.mean(), 3))

## Step 7 — Create and train the classifier
Logistic Regression is a classification algorithm despite the word 'Regression' in its name. The pipeline scales numeric features and then trains the classifier.

In [ ]:
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

model.fit(X_train, y_train)
print("Training complete!")

## Step 8 — Make predictions
`predict()` returns 0 (ham) or 1 (spam).

In [ ]:
y_pred = model.predict(X_test)

results = X_test.copy()
results["actual_is_spam"] = y_test
results["predicted_is_spam"] = y_pred
display(results.head(10))

## Step 9 — Evaluate the model
Accuracy is the share of correct predictions. Precision measures how many predicted spam examples were actually spam. Recall measures how much of all actual spam the model found. F1 balances precision and recall.

In [ ]:
print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1-score :", round(f1_score(y_test, y_pred, zero_division=0), 4))

print("\nDetailed classification report:")
print(classification_report(
    y_test, y_pred, target_names=["Ham (0)", "Spam (1)"], zero_division=0
))

## Step 10 — Plot a confusion matrix
The confusion matrix shows correct and incorrect predictions for both classes.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Ham", "Spam"])
display.plot(values_format="d")
plt.title("Confusion Matrix — Spam Detection")
plt.tight_layout()
plt.show()

## Step 11 — Understand feature influence
Because the numeric inputs are scaled inside the pipeline, these coefficients show the direction of each feature's relationship with the model prediction. They do not prove causation.

In [ ]:
classifier = model.named_steps["classifier"]
importance = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": classifier.coef_[0],
})
importance["absolute_coefficient"] = importance["coefficient"].abs()
importance = importance.sort_values("absolute_coefficient", ascending=False)

display(importance)

importance.sort_values("coefficient").plot(
    x="feature", y="coefficient", kind="barh", legend=False
)
plt.title("Logistic Regression Coefficients")
plt.xlabel("Coefficient (scaled features)")
plt.tight_layout()
plt.show()

## Step 12 — Save the trained model
The saved file lets you reuse the trained pipeline without training it again. Only load model files that you created or trust.

In [ ]:
import os\nos.makedirs("artifacts", exist_ok=True)
model_path = Path("artifacts/spam_classifier.joblib")
joblib.dump(model, model_path)
print(f"Saved model to: {model_path}")

## Step 13 — Predict one new example
These inputs are **features**, not raw message text. The values below are only an example and are not guaranteed to be spam.

In [ ]:
example = pd.DataFrame([{
    "num_links": 3,
    "num_words": 45,
    "has_offer": 1,
    "sender_score": 0.25,
    "all_caps": 1,
}])

prediction = model.predict(example)[0]
probabilities = model.predict_proba(example)[0]

label = "SPAM" if prediction == 1 else "HAM / NOT SPAM"
print("Prediction:", label)
print(f"Estimated spam probability: {probabilities[1]:.1%}")
print("\nReminder: this is a model estimate, not proof that a message is safe or fraudulent.")

## Step 14 — What to improve next

- Compare Logistic Regression with a Decision Tree or Random Forest.
- Add cross-validation and document the results.
- Add unit tests and a small prediction API.
- For genuine SMS-text classification, obtain a text dataset containing both the message and its spam/ham label, then build a TF-IDF + classifier pipeline.

**Do not claim that this notebook analyzes raw message text:** the supplied CSV contains only engineered numeric features.